# Zmiany wskazań poza zdarzeniami: czas trwania i kształt

Jak długie i jak duże są zmiany wskazań wagi poza zdarzeniami i na jakiej skali czasu przestaje je opisywać prosta albo parabola. Pytanie użytkownika z 2026-10-27: czy w czasie odsypu (30–60 s) zdarzają się zmiany, których korekcja dryfu trendem z okien obok nie obejmie. Zbiór: baza `2026-10-18 obiekt`, sesja 9, wariant `CURRENT_VARIANT`; fragmenty bez zdarzeń i zakłócenia z `dms_analysis.between_events_fragments`, obliczenia w `dms_analysis.weight_changes`.

1. **Wahnięcia:** wskazania wygładzone medianą ruchomą `SMOOTHING_S` (zakłócenia pominięte), na nich punkty zwrotne metodą zygzaka: ekstremum jest potwierdzone, kiedy wskazanie odejdzie od niego o co najmniej próg `THRESHOLDS_KG` w przeciwną stronę. Wahnięcie = odcinek od ekstremum do kolejnego ekstremum (jedno ramię „paraboli” albo pół okresu „sinusoidy”): czas trwania i zmiana wskazania. Pierwsze wahnięcie każdego ciągłego kawałka pomijane (jego początek ucina brzeg fragmentu), ostatnie niepotwierdzone ekstremum też.
2. **Odchyłka od prostej i paraboli:** fragmenty pocięte na rozłączne odcinki długości T (`SPANS_S`, próbki co ok. 1 s, odcinki z zakłóceniem pominięte); w każdym prosta i parabola dopasowane metodą najmniejszych kwadratów i odchylenie standardowe reszt. Przy samym szumie odczytu reszta nie zależy od T; jeśli rośnie z T, kształt na tej skali nie jest już prostą (parabolą).
3. **Pora dnia** wg zegara RPi, wg początku wahnięcia albo odcinka: noc `NIGHT_HOURS`, dzień `DAY_HOURS`, reszta „pozostałe”.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
from IPython.display import display

from dms_analysis.between_events_fragments import fragment_table, prepare_samples
from dms_analysis.prepared import load_session
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.snapshot import ANALYSIS_SESSION, ANALYSIS_SOURCE
from dms_analysis.weight_changes import (
    SMOOTHING_S,
    SwingBrowser,
    span_figure,
    span_residuals,
    span_summary,
    swing_duration_figure,
    swing_summary,
    swings,
)

DB_PATH = Path("../data/prepared/obiekt-sesja-9.db")
FIGURE_WIDTH = 1200

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

In [ ]:
session = load_session(DB_PATH, ANALYSIS_SOURCE, ANALYSIS_SESSION, CURRENT_VARIANT)
samples = prepare_samples(session)
fragments = fragment_table(samples)

swing_table = swings(samples)
residuals = span_residuals(samples)

## Wahnięcia

Tabela: dla każdego wygładzania i progu liczba wahnięć, wahnięcia na godzinę fragmentów, percentyle czasu trwania, udział wahnięć krótszych niż 30 / 60 / 120 s i zmiana wskazania. Mały próg przy krótkim wygładzaniu łapie głównie szum odczytu (działka 20 kg), więc ważne są wiersze z większym progiem.

In [ ]:
summary = swing_summary(swing_table, samples)
display(summary.round(1))

Rozkład skumulowany czasu wahnięć: ile procent wahnięć danej wielkości jest krótszych niż czas na osi X (oś logarytmiczna). Kolory = próg, linia = pora dnia.

In [ ]:
for smoothing_s in SMOOTHING_S:
    swing_duration_figure(swing_table, smoothing_s, FIGURE_WIDTH).show()

## Odchyłka od prostej i paraboli w zależności od długości odcinka

Mediana odchylenia reszt z odcinków danej długości. Ogony (90. i 99. percentyl) w tabeli — rzadkie odcinki z dużą zmianą kształtu są ważne dla największych błędów metod.

In [ ]:
display(span_summary(residuals).round(1))

span_figure(residuals, FIGURE_WIDTH).show()

## Przegląd wahnięć i dopasowań na wskazaniach

Wskazania fragmentu (bez zakłóceń), mediana ruchoma, linia zygzaka przez potwierdzone ekstrema oraz prosta i parabola dopasowane osobno w każdym odcinku długości T (te same odcinki co w tabeli i wykresie odchyłek; odcinek z zakłóceniem bez dopasowania). Na górze mediana odchylenia reszt w odcinkach tego fragmentu. Wybór wygładzania, progu, T („brak” = bez dopasowań) i numeru fragmentu; wykres można przybliżać (plotly-resampler dociąga próbki), serie włącza się i wyłącza w legendzie.

In [ ]:
SwingBrowser(samples, fragments, "Wahnięcia wskazań we fragmencie bez zdarzeń", FIGURE_WIDTH).widget